# CranioVision Preprocessing Sanity Checks

This notebook applies the MONAI pipelines from `src/data/transforms.py` to one case and checks tensor shapes, dtypes, and label values.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(REPO_ROOT)

In [ ]:
import numpy as np

from src.data.brats_dataset import discover_brats_cases
from src.data.transforms import build_train_transforms, build_val_transforms
from src.utils.config import load_config
from src.utils.visualization import plot_modalities_and_label

config = load_config(REPO_ROOT / 'configs' / 'default.yaml')

In [ ]:
cases = discover_brats_cases(
    root_dir=config['data']['root_dir'],
    modality_aliases=config['data']['modality_aliases'],
    label_aliases=config['data']['label_aliases'],
    file_extensions=config['data']['file_extensions'],
    expected_modalities=config['data']['expected_modalities'],
    validate_shapes=config['data']['validate_shapes'],
)
sample_case = cases[0]

In [ ]:
train_transforms = build_train_transforms(config)
val_transforms = build_val_transforms(config)

train_sample_list = train_transforms(sample_case)
train_sample = train_sample_list[0]
val_sample = val_transforms(sample_case)

print('Train image shape:', tuple(train_sample['image'].shape))
print('Train label shape:', tuple(train_sample['label'].shape))
print('Train label unique values:', np.unique(train_sample['label']))
print('Val image shape:', tuple(val_sample['image'].shape))
print('Val label shape:', tuple(val_sample['label'].shape))
print('Val label unique values:', np.unique(val_sample['label']))

In [ ]:
val_image = val_sample['image'].cpu().numpy()
val_label = val_sample['label'][0].cpu().numpy()
modality_arrays = {
    modality_name: val_image[idx]
    for idx, modality_name in enumerate(config['data']['expected_modalities'])
}

plot_modalities_and_label(
    modality_arrays=modality_arrays,
    label_array=val_label,
    case_id=sample_case['case_id'],
    axis=2,
)